# Query and Communication Complexity

## Two non-uniform models that bypass Turing machines and where almost every *provable* quantum-versus-classical separation actually lives. **Query complexity** counts oracle calls (Deutsch–Jozsa, Grover, Simon). **Communication complexity** counts bits/qubits exchanged between parties holding parts of an input.

# 1. The query model

## A **partial function** $f : D \to \{0, 1\}$ with $D \subseteq \{0,1\}^N$ is to be computed using black-box access to the input $x = x_1 \dots x_N$. An algorithm pays for each query to a single bit $x_i$. Everything else — internal computation, gates, classical post-processing — is free.

## Three standard measures for an input length $N$:

## - $D(f)$ = deterministic classical query complexity (worst-case over $x$, no randomness).
## - $R(f)$ = bounded-error randomised query complexity.
## - $Q(f)$ = bounded-error **quantum** query complexity (where the oracle is the unitary $|i\rangle|b\rangle \mapsto |i\rangle|b \oplus x_i\rangle$).

## "Bounded error" means succeeding with probability $\geq 2/3$ on every input.

## The basic separations from earlier in the course:

| Problem            | $D$         | $R$               | $Q$            |
|--------------------|-------------|-------------------|----------------|
| Deutsch–Jozsa      | $2^{n-1}+1$ | $O(1)$            | $1$            |
| Bernstein–Vazirani | $n$         | $\Omega(n)$       | $1$            |
| Simon              | $\Omega(2^{n/2})$ | $\Omega(2^{n/2})$ | $O(n)$    |
| Grover (OR)        | $N$         | $N$               | $\Theta(\sqrt N)$ |

## Simon and Grover give the cleanest *unconditional* quantum advantages we know — neither requires unproven conjectures.

# 2. Total functions: at most polynomial gap

## A **total** Boolean function is defined on every input in $\{0,1\}^N$. For total $f$, the quantum advantage is provably modest:

$$ \Large  D(f) \;\leq\; \mathrm{poly}\big(Q(f)\big). $$

## Beals–Buhrman–Cleve–Mosca–de Wolf (1998) gave $D(f) = O(Q(f)^6)$. Aaronson–Ben-David–Kothari–Rao–Tal (2021) tightened this to $D(f) = O(Q(f)^4)$ and showed the gap is tight: there exists a total $f$ with $D(f) = \tilde\Omega(Q(f)^4)$.

## Moral: exponential quantum speedups in the query model require a **promise** (a restricted input domain). For total problems, the most quantum computers can do is take a 4th root.

# 3. Lower-bound techniques

## Two main hammers for proving $Q(f) = \Omega(\text{something})$:

## ### The polynomial method (Beals et al., 1998)

## If a $T$-query quantum algorithm computes $f$ with bounded error, then $\Pr[\text{accept}]$ is a polynomial of degree $\leq 2T$ in the input bits $x_1, \dots, x_N$. So

$$ \Large  Q(f) \;\geq\; \tfrac{1}{2}\, \widetilde{\deg}(f), $$

## where $\widetilde{\deg}(f)$ is the *approximate degree* — the smallest degree of a real polynomial agreeing with $f$ to within $1/3$ on $\{0,1\}^N$. Bounding approximate degree is now a purely analytic question. This is how the $\Omega(\sqrt N)$ Grover lower bound is proved.

## ### The adversary method (Ambainis, 2000; the negative-weights variant by Høyer–Lee–Špalek)

## A general matrix-game/SDP characterisation of $Q(f)$. The **negative-weights adversary bound** is *tight*: for every $f$,

$$ \Large  Q(f) \;=\; \Theta(\mathrm{ADV}^\pm(f)). $$

## In principle, computing $\mathrm{ADV}^\pm(f)$ exactly characterises quantum query complexity. In practice it is an SDP that is rarely tractable by hand for interesting $f$.

## Together these methods give every quantum query lower bound we know.

In [1]:
# Empirically check the BBBV / Grover lower bound: for OR_N, classical R(OR) = Theta(N) and Q(OR) = Theta(sqrt N).
# Compare the worst-case classical search vs Grover for varying N.
import numpy as np
import math

def classical_search_expected_queries(N):
    # Random ordering of indices, marked at random; expected queries to find it = (N+1)/2
    return (N + 1) / 2

def grover_iterations(N, M=1):
    return int(np.floor((np.pi / 4) * np.sqrt(N / M)))

print(f'{"N":>8} | {"R(OR) ~":>10} | {"Q(OR) ~":>10} | {"speedup":>10}')
for k in range(4, 13):
    N = 2 ** k
    r = classical_search_expected_queries(N)
    q = grover_iterations(N)
    print(f'{N:>8} | {r:>10.1f} | {q:>10d} | {r/q:>10.2f}x')

       N |    R(OR) ~ |    Q(OR) ~ |    speedup
      16 |        8.5 |          3 |       2.83x
      32 |       16.5 |          4 |       4.12x
      64 |       32.5 |          6 |       5.42x
     128 |       64.5 |          8 |       8.06x
     256 |      128.5 |         12 |      10.71x
     512 |      256.5 |         17 |      15.09x
    1024 |      512.5 |         25 |      20.50x
    2048 |     1024.5 |         35 |      29.27x
    4096 |     2048.5 |         50 |      40.97x


# 4. Headline query-complexity results

## A few high-impact theorems beyond the early algorithms:

## - **OR / Grover search.** $Q(\mathrm{OR}_N) = \Theta(\sqrt N)$ (BBBV, 1997).
## - **AND-OR trees.** Balanced AND-OR tree of depth $d$ and fan-in 2 has $Q = \Theta(\sqrt N)$ (Reichardt, 2011, via   span programs / negative-weights adversary).
## - **Element distinctness.** $Q(\mathrm{ED}_N) = \Theta(N^{2/3})$ (Ambainis, upper; Aaronson–Shi, lower).
## - **Triangle finding.** $Q = \tilde O(N^{5/4})$ (Le Gall, 2014).
## - **Forrelation.** $Q = 1$, $R = \tilde\Omega(\sqrt N)$ (Aaronson–Ambainis, 2015) — the *largest* possible separation   between $Q$ and $R$ for a partial function.

## **Forrelation** is the witness for Raz–Tal's $\mathrm{BQP} \not\subseteq \mathrm{PH}$ oracle separation: a problem solvable with 1 quantum query, but requiring sub-exponential queries to compute by any constant-depth classical circuit with random oracle access.

# 5. Communication complexity — Yao's model

## Two players, Alice and Bob, each hold half of an input. Alice has $x \in \{0,1\}^n$, Bob has $y \in \{0,1\}^n$. They wish to compute $f(x, y) \in \{0, 1\}$ by exchanging messages. The cost is the **total number of bits** sent, in the worst case. Three measures, analogous to the query model:

## - $D(f)$ deterministic communication complexity,
## - $R(f)$ bounded-error randomised (with shared or private randomness),
## - $Q(f)$ bounded-error **quantum** communication (qubits exchanged).

## A useful related measure: $Q^*(f)$ where the parties may also start with an unlimited supply of **shared entanglement**.

## Trivially, $Q(f) \leq R(f) \leq D(f) \leq n + 1$ (Alice sends her entire input).

# 6. The three benchmark problems

## ### EQUALITY: $f(x, y) = [x = y]$

## - $D(\mathrm{EQ}) = n + 1$.
## - $R(\mathrm{EQ}) = \Theta(\log n)$ with shared randomness (hash and send the hash).
## - $Q(\mathrm{EQ}) = \Theta(\log n)$ — **same as classical**. No quantum win.

## ### DISJOINTNESS: $f(x, y) = [\exists i : x_i = y_i = 1]$

## - $D, R = \Theta(n)$ (the canonical NP-style problem in communication complexity).
## - $Q(\mathrm{DISJ}) = \Theta(\sqrt n)$ (Aaronson–Ambainis upper, Razborov lower) — *quadratic* quantum speedup.
## - Crucial because DISJ shows quantum protocols *can* beat classical even for total functions, by exactly $\sqrt n$.

## ### INNER PRODUCT mod 2: $f(x, y) = \langle x, y \rangle \bmod 2$

## - $D, R, Q = \Theta(n)$ — no quantum speedup at all (Cleve–van Dam–Nielsen–Tapp).

## So total Boolean functions in communication complexity also have at most a polynomial quantum advantage. Exponential separations require **promises** or **relations** (search-style problems), exactly like in query complexity.

# 7. Exponential separations and Raz's problem

## **Raz's problem** (1999): a promise problem $f$ with

$$ \Large  Q(f) = O(\log n), \qquad R(f) = \Omega(n^{1/4}). $$

## This was the first super-polynomial separation in communication complexity. Later strengthened by Bar-Yossef–Jayram–Kerenidis (2004) and Gavinsky–Kempe–Kerenidis–Raz–de Wolf (2007) to exponential gaps for *one-way* and *simultaneous-message* models.

## Klartag–Regev (2011) extended this to a problem where even *one-way* quantum communication is exponentially smaller than any randomised protocol.

## The takeaway: in the communication setting, the gulf between $Q$ and $R$ can be *enormous*, but only for promise problems.

# 8. Entanglement as a resource

## What does *shared entanglement* buy in communication?

## - **Superdense coding** (chapter 3): one ebit + one qubit ⇒ two classical bits.   So shared entanglement halves the cost of sending classical messages.
## - **Teleportation** (chapter 3): one ebit + two classical bits ⇒ one qubit.   Free entanglement turns classical channels into quantum ones.

## In communication complexity, the entanglement-assisted class $Q^*$ is *not known* to be more powerful than $Q$ for total functions, and the gap (if any) is at most polynomial. For promise problems, however, the difference can be unbounded — entanglement is genuinely useful for some interactive tasks.

## Bell inequality violations are themselves communication-complexity statements: a Bell game is a one-round, no-communication problem where quantum strategies beat classical ones by a measurable margin.

# 9. Why these models matter

## - **All provable quantum speedups** for natural problems live here. BQP-vs-BPP is open; $Q$-vs-$R$ has unconditional separations.
## - Communication complexity gives the **only known unconditional lower bounds** on the size of streaming algorithms,   data structures, and a sweep of other concrete computational models.
## - Quantum query complexity feeds back into circuit lower bounds via the polynomial method (e.g. AC$^0$ lower bounds   from approximate-degree results).
## - The same techniques that prove $Q(\mathrm{OR}_N) \geq \sqrt N$ underlie the **BBBV theorem** — telling us Grover is optimal,   and therefore that quantum computers cannot solve NP-hard problems by brute force exponentially faster than classical.

# Recap

## - **Query complexity** $D, R, Q$: counts black-box accesses.
## - For **total** $f$: $D(f) = O(Q(f)^4)$. Exponential speedups need a **promise**.
## - **Polynomial method** and **adversary method** are the two lower-bound hammers; the negative-weights adversary is tight.
## - **Communication complexity** Yao model: Alice/Bob exchange bits or qubits.
## - **DISJ**: $\sqrt n$ quantum speedup. **Raz's problem**: exponential gap for a promise problem.
## - Shared entanglement: matters for promise problems, at most polynomially for total ones.

## Next: the *no-go* side of the story — the no-cloning, no-deletion, no-broadcasting, and no-signalling theorems that constrain what *any* quantum algorithm can ever achieve.